<style>
    @import url('https://fonts.googleapis.com/css2?family=Oswald:wght@400;600&display=swap');
    h1.course-title {
        font-family: 'Oswald', sans-serif;
        font-size: 2.4em;
        color: #E7C173;
        letter-spacing: 0.05em;
        border-bottom: 2px solid #E7C173;
        padding-bottom: 0.3em;
        margin-bottom: 0.2em;
    }
    h2.course-subtitle { font-family: 'Oswald', sans-serif; color: #aaa; font-size: 1.2em; }
</style>

<h1 class='course-title'>MACHINE LEARNING IN INDUSTRY</h1>
<h2 class='course-subtitle'>Cardo AI · MSCA Digital Doctoral Network · Day 4 Workshop</h2>

# Day 4 Workshop — MLOps Hands-On Exercise

**Time:** 1:30–3:00 PM (90 minutes for Steps 1–3; Step 4 is a stretch goal)

## What you will do

| Step | Task | Reference |
|---|---|---|
| **1** | Log a training run to MLflow and register the model | `notebooks/01_mlflow_tracking.ipynb` |
| **1e** | Subgroup performance analysis | `notebooks/01_mlflow_tracking.ipynb`, Section 7 |
| **2** | Run Evidently drift detection and log the report | `notebooks/02_drift_monitoring.ipynb` |
| **2e–g** | PSI from scratch, multiple testing correction, concept drift experiment | `notebooks/02_drift_monitoring.ipynb`, Sections 6–9B |
| **2h** | Adversarial drift injection (failure exercise) | — |
| **3** | Build the Docker image and query the prediction API | `day4/Dockerfile` + `day4/README.md` |
| **4** *(stretch)* | Trigger the GitHub Actions CI/CD workflow on a fork | `.github/workflows/ml-pipeline.yml` |

**Rules:**
- Each `TODO` has a reference comment pointing to the relevant notebook section.
- The skeleton runs without errors even with TODOs left blank — you add functionality incrementally.
- Work in your group. Discuss before you type.

> **Tip:** If you get stuck, the complete solution lives in the lecture notebooks and `day4/src/`. Use them — that's what they're there for.

---
## Setup — Run this first (do not modify)

In [ ]:
# ── All imports pre-filled — no time wasted here ─────────────────────────────
import sys, time, json, tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset

repo_root = Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from day4.src.train import (
    DATA_PATH, SEED, TARGET_BIN_COL,
    build_pipeline, get_feature_columns, load_data, split_data,
)

np.random.seed(SEED)
print("Setup complete.")

In [ ]:
# ── Data loading — pre-filled, just run it ────────────────────────────────────
data_path = repo_root / DATA_PATH
df = load_data(data_path)
train_df, val_df, test_df = split_data(df)

numeric_cols, categorical_cols = get_feature_columns(train_df)
feature_cols = numeric_cols + categorical_cols

X_train, y_train = train_df[feature_cols], train_df[TARGET_BIN_COL]
X_val,   y_val   = val_df[feature_cols],   val_df[TARGET_BIN_COL]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(test_df)} rows")
print(f"Features: {len(feature_cols)}")

---
## Step 1 — Experiment Tracking with MLflow

**Goal:** Log a training run (params, metrics, model) to a local MLflow server. Register the model.

**Prerequisite:** Start the MLflow server in a separate terminal:
```bash
make -f day4/Makefile mlflow-server
```

> **Reference:** `notebooks/01_mlflow_tracking.ipynb`, Sections 4–7

In [ ]:
# ── 1a. Configure MLflow ──────────────────────────────────────────────────────

# TODO: Set the tracking URI to http://127.0.0.1:5000
# See: notebook 01, Section 4 — mlflow.set_tracking_uri(...)
mlflow.set_tracking_uri(...)  # ← your code here

# TODO: Create or select an experiment named "workshop-adult-income"
# See: notebook 01, Section 4 — mlflow.set_experiment(...)
mlflow.set_experiment(...)    # ← your code here

print(f"Tracking URI : {mlflow.get_tracking_uri()}")

In [ ]:
# ── 1b. Define hyperparameters ────────────────────────────────────────────────

# TODO: Experiment with different values (see Day 2 results for guidance).
# These defaults match the best config from Day 2 — try beating them.
params = {
    "n_estimators":  300,   # TODO: try different values
    "learning_rate": 0.1,   # TODO: try different values
    "max_depth":     5,     # TODO: try different values
    "num_leaves":    31,    # TODO: try different values
}

print("Params:", params)

In [ ]:
# ── 1c. Train and log ─────────────────────────────────────────────────────────

# TODO: Complete the MLflow run.
# Inside the with-block:
#   1. Log params with mlflow.log_params(params)
#   2. Train: pipe = build_pipeline(numeric_cols, categorical_cols, **params); pipe.fit(...)
#   3. Compute val_auc, val_f1, val_accuracy and log with mlflow.log_metrics({...})
#   4. Log the model with mlflow.sklearn.log_model(pipe, artifact_path="model")
# See: notebook 01, Section 5

workshop_run_id = None

with mlflow.start_run(run_name="workshop-run") as run:
    workshop_run_id = run.info.run_id

    # ↓↓↓ your code here ↓↓↓


    # ↑↑↑ your code here ↑↑↑
    pass  # remove this once you've added your code

print(f"Run ID: {workshop_run_id}")
print("Open http://127.0.0.1:5000 and find your run in 'workshop-adult-income'.")

In [ ]:
# ── 1d. Register the model ────────────────────────────────────────

# TODO: Register your model in the MLflow Model Registry and set the @champion alias.
# See: notebook 01, Section 7
#
# Steps:
#   1. mv = mlflow.register_model(model_uri=f"runs:/{workshop_run_id}/model", name="adult-income-classifier")
#   2. client = mlflow.tracking.MlflowClient()
#   3. client.set_registered_model_alias("adult-income-classifier", "champion", mv.version)
#
# The @champion alias is important — it's how make api, make drift-check,
# and docker-compose find the model. Without it, those tools will fail.

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 1e. Subgroup Performance ──────────────────────────────────────────────────
# Reporting per-subgroup metrics is a minimum for responsible deployment.
# See: Barocas, Hardt & Narayanan (2019) "Fairness and Machine Learning"
#
# TODO: Compute per-group AUC for the "marital_status" column.
# See: notebook 01, Section 7 — "Model Diagnostics — Calibration and Fairness"
#
# Steps:
#   1. Use the pipeline trained in 1c (or retrain one)
#   2. Get y_proba on the validation set
#   3. For each unique value in val_df["marital_status"]:
#        mask = val_df["marital_status"] == group
#        group_auc = roc_auc_score(y_val[mask], y_proba[mask])
#   4. Print results as a table
#   5. Log each group AUC to MLflow (e.g., mlflow.log_metric(f"val_auc_{group}", ...))
#
# Discussion: Is the gap between groups meaningful? What would you do about it?

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

**Step 1 self-check:**
- [ ] My run appears in the MLflow UI under experiment `workshop-adult-income`
- [ ] The run has logged params, metrics (including `val_auc`), and a model artifact
- [ ] (Optional) The model is registered in the Model Registry
- [ ] I computed per-subgroup AUC and can discuss the fairness implications

---
## Step 2 — Drift Detection with Evidently

**Goal:** Build reference/analysis sets, run univariate drift detection, log the report to MLflow.

> **Reference:** `notebooks/02_drift_monitoring.ipynb`, Sections 5–7

In [ ]:
# ── 2a. Re-train a model (needed for scoring) ─────────────────────────────────
# Pre-filled — just run it.
default_params = {"n_estimators": 300, "learning_rate": 0.1, "max_depth": 5}
pipe = build_pipeline(numeric_cols, categorical_cols, **default_params)
pipe.fit(X_train, y_train)
print("Model trained.")

In [ ]:
# ── 2b. Build reference and analysis sets — pre-filled ────────────────────────
# You do not need to change this cell.
# Note: we use train_df (from split_data) — NOT df[df["split"] == "train"] —
# because split_data performs entity-aware deduplication by person_id and
# holds out the validation set. This matches the Day 1/Day 2 best practice.

reference_df = train_df[feature_cols + [TARGET_BIN_COL]].copy()
reference_df["y_pred_proba"] = pipe.predict_proba(reference_df[feature_cols])[:, 1]
reference_df["y_pred"] = (reference_df["y_pred_proba"] >= 0.5).astype(int)

analysis_df = test_df[feature_cols].copy()
analysis_df["y_pred_proba"] = pipe.predict_proba(analysis_df[feature_cols])[:, 1]
analysis_df["y_pred"] = (analysis_df["y_pred_proba"] >= 0.5).astype(int)

print(f"Reference: {len(reference_df)} rows | Analysis: {len(analysis_df)} rows")

In [ ]:
# ── 2c. Univariate drift detection ────────────────────────────────────────────

# TODO: Run Evidently DataDriftPreset on reference and analysis sets.
# See: notebook 02, Section 7
#
# Steps:
#   1. Create a DataDefinition with numerical_columns and categorical_columns
#   2. Wrap reference_df[feature_cols] and analysis_df[feature_cols] as Evidently Datasets
#   3. Create a Report with DataDriftPreset()
#   4. Run the report and save the HTML to a temp file
#   5. Parse result.dict() to extract per-feature drift statistics
#
# Note: With > 1000 rows, Evidently defaults to Wasserstein/Jensen-Shannon
# distances (threshold=0.1). Drift is detected when distance >= threshold.

# ↓↓↓ your code here ↓↓↓

drift_result = None  # replace with your Evidently report result

# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 2d. Log Evidently report to MLflow ────────────────────────────────────────

# TODO: Log the drift report HTML as an MLflow artifact.
# See: notebook 02, Section 10

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 2e. Compute PSI from Scratch ─────────────────────────────────────────
# PSI (Population Stability Index) is the industry-standard metric for
# distribution shift. Regulators (ECB, Fed SR 11-7) and model validation
# teams use it as the primary drift signal.
#
# TODO: Implement PSI and compute it for all numeric features + scores.
# See: notebook 02, Section 6 — "PSI — The Industry Standard for Drift Detection"
#
# PSI formula:
#   PSI = Σ (p_analysis_i - p_reference_i) × ln(p_analysis_i / p_reference_i)
#
# Steps:
#   1. Write a function compute_psi(reference, analysis, n_bins=10):
#        - Compute quantile-based bin edges from reference
#        - Histogram both arrays into those bins
#        - Convert counts to proportions, clip to avoid log(0)
#        - Return PSI value
#   2. Compute score PSI: compute_psi(reference_df["y_pred_proba"], analysis_df["y_pred_proba"])
#   3. Compute feature PSI for each numeric feature
#   4. Print results with thresholds: < 0.10 stable, 0.10–0.25 investigate, > 0.25 retrain
#
# Compare: How do PSI results align with Evidently's drift test results from 2c?

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 2f. Multiple Testing Correction ───────────────────────────────────────────
# Bonferroni correction applies to **p-value-based** tests, NOT distance-based
# methods. Our default Evidently run (2c) uses Wasserstein/Jensen-Shannon
# distances — there's no p-value to correct.
#
# To demonstrate Bonferroni, re-run Evidently with classical tests that
# return p-values: KS for numeric, chi-squared for categorical.
# See: notebook 02, Section 7B
#
# TODO: Apply Bonferroni correction to p-value-based drift tests.
#
# Steps:
#   1. Re-run Evidently with p-value tests:
#        Report(metrics=[DataDriftPreset(num_method="ks", cat_method="chisquare")])
#      (suppress RuntimeWarning for divide-by-zero on near-constant features)
#   2. Extract p-values from the new report's result.dict()
#   3. Count n_features; compute alpha_corrected = 0.05 / n_features
#   4. Print uncorrected vs. corrected alpha
#   5. Count features with p < 0.05 (uncorrected) vs. p < alpha_corrected (Bonferroni)
#   6. How many alerts are lost after correction?
#
# Discussion: Over a year with 19 features checked nightly,
# expect ~347 false alerts at alpha=0.05 uncorrected.
# What are alternatives to Bonferroni? (Hint: Benjamini-Hochberg FDR)

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 2g. Concept Drift Experiment ──────────────────────────────────────────────
# All input-distribution-based monitoring (PSI, KS, chi-squared)
# monitors P(X) or P(scores). Concept drift changes P(Y|X) — the features
# and scores look the same but the outcome relationship has changed.
#
# TODO: Simulate concept drift and show that PSI and KS are unchanged.
# See: notebook 02, Section 9B — "Concept Drift — When Input Monitoring Fails"
#
# Steps:
#   1. Start from analysis_df (which has y_pred_proba from cell 2b)
#   2. Create a copy with ground truth: add the real test labels
#   3. Compute score PSI and KS BEFORE the label flip (baseline values)
#   4. Flip labels for high-confidence predictions (y_pred_proba > 0.7)
#      This simulates: the model is confident, but the world has changed
#   5. Compute score PSI and KS AFTER the label flip
#      → both should be IDENTICAL to Step 3 (scores haven't changed)
#   6. Compute actual AUC before and after the label flip
#      → AUC should drop significantly after the flip
#   7. Print all values side by side (before vs. after)
#
# Key insight: PSI and KS are **unchanged** by concept drift — they return
# identical values before and after the label flip, because they monitor
# P(scores), not P(Y|X). Label collection is non-negotiable in production.
# See: Gama et al. (2014) "A Survey on Concept Drift Adaptation"

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 2h. Adversarial Drift Injection ──────────────────────────────────────────
# Not all feature drift affects performance equally. Some features can shift
# dramatically without impacting predictions. This exercise builds intuition
# for which drift signals are actionable.
#
# TODO: Inject artificial drift into the analysis set and measure the impact.
#
# Steps:
#   1. Create a corrupted copy of analysis_df
#   2. Corrupt "education": set 30% of rows to "PhD"
#        idx = corrupted_df.sample(frac=0.3, random_state=42).index
#        corrupted_df.loc[idx, "education"] = "PhD"
#   3. Corrupt "capital_gain": add Gaussian noise (mean=0, std=5000)
#        corrupted_df["capital_gain"] += np.random.normal(0, 5000, len(corrupted_df))
#   4. Re-score: get new y_pred_proba from pipe.predict_proba(corrupted_df[feature_cols])
#   5. Run univariate drift on the corrupted set (or compute PSI per feature)
#   6. Compute score PSI (reference y_pred_proba vs. corrupted y_pred_proba)
#   7. Compute actual AUC on corrupted predictions vs. original test labels
#
# Questions:
#   - Which features does drift detection flag?
#   - Does the score distribution change? By how much (PSI)?
#   - Does AUC actually degrade? By how much?
#   - Key insight: not all feature drift is equally dangerous.
#     Some features are low-importance → drift is noise.
#     Feature importance (from Step 1) tells you which drift to worry about.

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 2i. (Stretch) Domain Classifier — Multivariate Drift ─────────────────────
# Univariate tests check each feature independently. A domain classifier
# tests the JOINT distribution: can a model tell reference from current?
# AUC ≈ 0.5 → no drift.  AUC >> 0.5 → drift detected.
# This is a single test — no multiple testing correction needed.
# See: notebook 02, Section 7C; Rabanser et al. (2019) "Failing Loudly"
#
# TODO: Implement a domain classifier for multivariate drift detection.
#
# Steps:
#   1. Combine reference_df[feature_cols] and analysis_df[feature_cols]
#   2. Create labels: 0 = reference, 1 = current
#   3. Build a Pipeline: ColumnTransformer(
#          numerics: SimpleImputer(strategy="median") + StandardScaler,
#          categoricals: SimpleImputer(strategy="constant", fill_value="MISSING") + OneHotEncoder
#      ) + LogisticRegression(max_iter=1000)
#      Note: SimpleImputer is required — the data has missing values.
#   4. Evaluate with 5-fold cross_val_score(scoring="roc_auc")
#   5. Print the AUC. Is it close to 0.5?
#   6. Re-run on your adversarial data from 2h. Does AUC increase?
#      Hint: if 2h perturbed numeric columns into categorical ones,
#      cast categorical columns to str before concatenating.
#   7. Fit on the adversarial data and inspect clf.coef_ — which features
#      does the classifier use to distinguish the datasets?
#      Hint: aggregate |coef| across OHE dummy columns back to original features.
#
# Discussion: How does this compare to the per-feature results from 2c?
# Does the domain classifier catch drift that univariate tests missed (or vice versa)?

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

In [ ]:
# ── 2j. (Stretch) Multivariate Wasserstein via PCA ───────────────────────────
# Compute a single Wasserstein distance over the joint feature distribution
# by reducing dimensionality first (high-D Wasserstein is computationally
# expensive and statistically unreliable).
# See: Rabanser et al. (2019) — dimensionality reduction before testing helps.
#
# TODO: Compute a multivariate drift distance using PCA + Wasserstein.
#
# Steps:
#   1. Preprocess: scale numerics, one-hot encode categoricals (same as 2i)
#   2. Fit PCA(n_components=5) on the reference set
#   3. Transform both reference and current into the 5-D PCA space
#   4. Compute scipy.stats.wasserstein_distance on each PCA component
#   5. Average (or max) across components for a single multivariate distance
#   6. Use a permutation test (shuffle labels 100 times, recompute distance)
#      to get a p-value for the observed distance
#   7. Compare: does this agree with the domain classifier from 2i?
#
# Discussion: What are the tradeoffs between the domain classifier and
# the PCA + Wasserstein approach? When would you prefer one over the other?

# ↓↓↓ your code here ↓↓↓


# ↑↑↑ your code here ↑↑↑

**Step 2 self-check:**
- [ ] Evidently drift report is displayed
- [ ] I can identify which features (if any) drifted
- [ ] The HTML report is logged as an MLflow artifact
- [ ] I implemented PSI from scratch and compared results to Evidently's statistical tests
- [ ] I computed the Bonferroni-corrected alpha and know how many alerts survived
- [ ] I demonstrated that concept drift is invisible to PSI and KS
- [ ] I injected adversarial drift and can explain why not all drift degrades performance
- [ ] (Stretch) Domain classifier AUC is ~0.5 on clean data; higher on adversarial data
- [ ] (Stretch) PCA + Wasserstein distance agrees with domain classifier results

---
## End-of-Workshop Checklist

Before Day 5, confirm that your group has:

- [ ] **MLflow:** At least one training run logged with params, metrics, and a model artifact
- [ ] **MLflow:** The model registered in the Model Registry (any stage)
- [ ] **Fairness:** Per-subgroup AUC computed and discussed
- [ ] **Evidently:** Drift detection ran on reference/analysis sets; report logged to MLflow
- [ ] **PSI:** Implemented from scratch; compared results with Evidently's statistical tests
- [ ] **Multiple Testing:** Bonferroni correction applied; false alert rate understood
- [ ] **Concept Drift:** Simulated label flip; confirmed PSI and KS are unchanged by concept drift
- [ ] **(Stretch) Domain Classifier:** Multivariate drift via domain classifier; feature importances inspected
- [ ] **(Stretch) Multivariate Distance:** PCA + Wasserstein distance with permutation test
- [ ] **Docker:** The container started successfully and returned a prediction from `/predict`
- [ ] **Discussion:** Your group has decided which MLOps practices you'll incorporate into your project (minimum: experiment tracking + one drift check)
- [ ] **GitHub Actions:** Workflow triggered and artifacts downloaded

---

**More food for thought (3:00–4:30 PM):**

Transition to group project time. Discuss:
1. What will your reference and analysis sets be? (What's the equivalent of the train/test split in your project data?)
2. Which features are most likely to drift in your domain?
3. What will a drift alert trigger in your project? (Retrain? Alert? Manual review?)
4. How will you collect labels? What is the delay between prediction and label availability?
5. Which monitoring metric matters most for your domain — PSI, KS, or something else?

---
## Step 3 — Docker

**Goal:** Build the prediction service container and get a live prediction from it.

This step happens **in a terminal**, not in this notebook.

### Instructions

From the **repo root**:

```bash
# Build the Docker image (requires Step 1 to be complete)
# run `make -f day4/Makefile mlflow-server`, then in a separate terminal, run `make -f day4/Makefile train`
make -f day4/Makefile docker-build

# Run the container
make -f day4/Makefile docker-run
```

In a **second terminal**, test it:

```bash
# Health check
curl http://localhost:8080/health

# Prediction (copy-paste this)
curl -X POST http://localhost:8080/predict \
     -H "Content-Type: application/json" \
     -d '{
           "age": 35,
           "workclass": "Private",
           "education": "Bachelors",
           "education_num": 13,
           "marital_status": "Married-civ-spouse",
           "occupation": "Prof-specialty",
           "relationship": "Husband",
           "race": "White",
           "sex": "Male",
           "capital_gain": 0,
           "capital_loss": 0,
           "hours_per_week": 45,
           "native_country": "United-States"
         }'
```

You can also explore the API docs at **http://localhost:8080/docs**.

### Questions to discuss with your group

1. The Dockerfile bakes the model artifact (`day4/outputs/model/`) into the image at build time via `COPY`. What are the trade-offs of this approach vs. pulling the model from a remote MLflow server at container startup?
2. The container loads the model from a local path (`MODEL_URI="/app/model"`). What would you change to load from the Model Registry instead (e.g., `models:/adult-income-classifier@champion`)?
3. The container exposes a single prediction endpoint. In production, what other endpoints would you want? (Hint: batch scoring, metrics, model version info)

In [ ]:
# ── Optional: call the API from the notebook ──────────────────────────────────
# (requires the container to be running: make docker-run)
import urllib.request, urllib.error, json as _json

SAMPLE_PAYLOAD = {
    "age": 35, "workclass": "Private", "education": "Bachelors",
    "education_num": 13, "marital_status": "Married-civ-spouse",
    "occupation": "Prof-specialty", "relationship": "Husband",
    "race": "White", "sex": "Male",
    "capital_gain": 0, "capital_loss": 0,
    "hours_per_week": 45, "native_country": "United-States",
}

try:
    req = urllib.request.Request(
        "http://localhost:8080/predict",
        data=_json.dumps(SAMPLE_PAYLOAD).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=3) as resp:
        result = _json.loads(resp.read())
    print(f"Prediction : {result['label']} (p={result['probability']:.4f})")
    print(f"Model URI  : {result['model_uri']}")
except urllib.error.URLError:
    print("Container not running. Start it with: make -f day4/Makefile docker-run")

**Step 3 self-check:**
- [ ] `curl http://localhost:8080/health` returns `{"status": "ok", "model_loaded": true}`
- [ ] The `/predict` endpoint returns a JSON response with `prediction`, `probability`, and `label`
- [ ] I can explain why the model is baked into the image (`MODEL_URI="/app/model"`) and what would change in a production setup

---
## Step 4 — CI/CD with GitHub Actions

**Goal:** Trigger the pre-built ML pipeline workflow on your own fork and inspect the artifacts.

This step is **observational** — no code to write. It's about understanding how the CI/CD system works.

### Instructions

1. **Fork** the course repository on GitHub.
2. **Clone** your fork locally (or push from your existing clone).
3. Make a small change to any file in `day4/` (e.g., add a comment to `train.py`) and `git push`.
4. Navigate to the **Actions** tab on your fork on GitHub.
5. Watch the `ML Pipeline` workflow run. Expand each step to see the logs.
6. Once complete, click **Summary** → download the artifacts:
   - `mlflow-run-<sha>` — the full MLflow run directory (explore it locally with `mlflow ui`)
   - `drift-report-<sha>` — open `drift_report.html` in your browser

### Read the YAML

Open `.github/workflows/ml-pipeline.yml` and answer these questions:

1. What is `MLFLOW_TRACKING_URI` set to in the workflow? Why is this different from your local setup?
2. Why does the drift-check step use `continue-on-error: true`?
3. Which step would cause the workflow to fail (exit code 1) if drift exceeds the threshold?
4. How would you modify the workflow to also build and push the Docker image to a container registry?
5. What would you need to change to make this run on a schedule (e.g., every night at 2 AM)?

**Step 4 self-check:**
- [ ] The GitHub Actions workflow ran successfully on my fork
- [ ] I downloaded and inspected both artifact archives
- [ ] I can answer all five questions above
- [ ] I understand why the CI uses `file:./mlruns` (relative, file-based tracking) and how a production setup would differ